## MIE360 Fall 2026 Project: Hospital Modeling and Simulation
### Deliverable 3.2: Base Model

Signatures
|Student|ID|
|-------|--|
|Gabriel Faustino|1011376823|
|Sunny Wu|1010978239|
|Benjamin Zhou|1011085772|

**Note: It is recommended to use a virtual environment (venv) before installing the required modules.** Run `python -m venv .venv` and `./.venv/Scripts/activate` before installing the required modules.

**Note: this notebook is designed to run with the locally-installed CSV data files after cloning.** Run `pip install -r ./deps/requirements.txt` before running.

In [67]:
import pandas as pd
import numpy as np
from scipy import stats
import simpy
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from enum import Enum
import random
import typing
import math

In [68]:
# Time unit helper functions
def days_to_hours(d: float) -> float:
    return d * 24


def days_to_minutes(d: float) -> float:
    return d * 24 * 60


def hours_to_days(h: float) -> float:
    return h / 24


def hours_to_minutes(h: float) -> float:
    return h * 60


def minutes_to_days(m: float) -> float:
    return m / 60 / 24


def minutes_to_hours(m: float) -> float:
    return m / 60


def fraction_of_day(d: float) -> float:
    return d % 1


def hour_of_day(h: float) -> float:
    return h % 24


def minute_of_day(m: float) -> float:
    return m % (24 * 60)


def next_day(d: float) -> float:
    return math.ceil(d)


def next_day_hour(h: float) -> float:
    return days_to_hours(next_day(hours_to_days(h)))


def next_day_minute(m: float) -> float:
    return days_to_minutes(next_day(minutes_to_days(m)))

In [69]:
# Load treatment time MLEs
df_treatment_mles = pd.read_csv("./out/treatment_distribution_fits.csv", index_col="distribution_specialty")
display(df_treatment_mles)

,p,shape,loc,scale
distribution_specialty,,,,
Gamma-Gastro_Endo,0.886080,0.846696,0.0,3.986304
Gamma-Ortho,0.321779,1.014737,0.0,5.411163
Gamma-Renal,0.674105,0.512199,0.0,9.416783
Exponential-Ortho,0.379521,NaN,0.0,5.490906
Lognormal-Card,0.175105,0.986174,0.0,1.777177
Lognormal-Respi,0.280705,1.133348,0.0,1.717883


In [70]:
# Load hourly arrival Poisson MLEs
df_arrival_mles = pd.read_csv("./out/arrival_poisson_fits.csv", index_col="hour")
display(df_arrival_mles)

,lambda
hour,
1am,1.61
2am,1.25
3am,0.98
4am,0.67
5am,0.53
6am,0.49
7am,0.95
8am,1.24
9am,1.68


In [71]:
# Align CSV (starts at 1AM, ends at 12AM) with simulation times (starts at 0AM, ends at 11PM)
hour_labels = (['12am'] + [f'{h}am' for h in range(1,12)] + ['12pm'] + [f'{h}pm' for h in range(1,12)])
hourly_rates = df_arrival_mles.loc[hour_labels, 'lambda'].to_numpy()

In [ ]:
class TreatmentType(Enum):
    SURG = ("Surg", 0.22, stats.expon(scale=1/0.4237))
    CARD = ("Card", 0.18, stats.lognorm(s=df_treatment_mles.loc["Lognormal-Card"]["shape"], scale=df_treatment_mles.loc["Lognormal-Card"]["scale"]))
    GEN_MED = ("Gen Med", 0.16, stats.lognorm(scale=np.exp(0.921), s=0.949))
    ORTHO = ("Ortho", 0.14, stats.expon(scale=df_treatment_mles.loc["Exponential-Ortho"]["scale"]))
    GASTRO_ENDO = ("Gastro-Endo", 0.08, stats.gamma(a=df_treatment_mles.loc["Gamma-Gastro_Endo"]["shape"], scale=df_treatment_mles.loc["Gamma-Gastro_Endo"]["scale"]))
    ONCO = ("Onco", 0.07, stats.gamma(a=0.612, scale=9.689))
    NEURO = ("Neuro", 0.06, stats.lognorm(scale=np.exp(0.53), s=1.133))
    RENAL = ("Renal", 0.05, stats.gamma(a=df_treatment_mles.loc["Gamma-Renal"]["shape"], scale=df_treatment_mles.loc["Gamma-Renal"]["scale"]))
    RESPI = ("Respi", 0.04, stats.lognorm(s=df_treatment_mles.loc["Lognormal-Respi"]["shape"], scale=df_treatment_mles.loc["Lognormal-Respi"]["scale"]))

    def __new__(cls, name: str, proportion: float, treatment_time_rv):
        obj = object.__new__(cls)
        obj._value_ = name
        obj.type_name = name
        obj.proportion = proportion
        obj.treatment_time_rv = treatment_time_rv
        return obj

    @classmethod
    def list(cls):
        return [t for t in cls]

    @classmethod
    def weightings(cls):
        return [t.proportion for t in cls]

    @classmethod
    def random_type(cls):
        return random.choices(cls.list(), weights=cls.weightings())[0]


class Ward(object):

    def __init__(self, env: simpy.Environment,
                 ward_id: int,
                 treatment_type: set[TreatmentType],
                 bed_count: int,
                 physician_count: int,
                 physician_discharge_time: float,
                 physician_discharge_interval: tuple[float, float], # [3PM, 5PM]
                 cleaner_time_rv):
        self.env = env
        self.ward_id = ward_id
        self.treatment_type = treatment_type
        self.physician_discharge_time = physician_discharge_time
        self.physician_discharge_interval = physician_discharge_interval
        self.cleaner_time_rv = cleaner_time_rv

        # Ward resources
        self.beds = simpy.Resource(env, capacity=bed_count)
        self.physicians = simpy.Resource(env, physician_count)

        self.num_occupied_beds = 0 # patient in bed
        self.num_unavailable_beds = 0 # beds occupied or being cleaned
        self.queue_log = [(env.now, 0)]
        self.occupied_bed_log = [(env.now, 0)]
        self.unavailable_bed_log = [(env.now, 0)]
        self.patient_records = []

    # LOGGING METHODS
    def log_queue(self):
        return self.queue_log.append((self.env.now, len(self.beds.queue)))

    def log_occupied_beds(self):
        return self.occupied_bed_log.append((self.env.now, self.num_occupied_beds))
    
    def log_unavailable_beds(self):
        return self.unavailable_bed_log.append((self.env.now, self.num_unavailable_beds))

    # HELPER FOR METRICS
    def time_weighted_average(self, log, start=0, end=None):
        if end is None:
            end = self.env.now

        if not 0 <= start < end <= self.env.now:
            raise ValueError('Invalid observation period')

        if not log or start < log[0][0]:
            raise ValueError('The log must cover the observation period')
        
        total = 0
        for i in range(len(log)):
            # value represents queue length
            current_time, value = log[i]
            if i + 1 < len(log):
                next_time = log[i+1][0]
            else:
                next_time = end

            # only consider observations within specified time period
            interval_start = max(current_time, start)
            interval_end = min(next_time, end)

            if interval_end > interval_start:
                total += value * (interval_end-interval_start)
            
        return total / (end-start)

    def get_waiting_records(self, start=0, end=None):
        if end is None:
            end = self.env.now
        if not 0 <= start < end <= self.env.now:
            raise ValueError('Invalid observation period')
        selected_records = [r for r in self.patient_records if start <= r['request_time'] < end]
        for r in selected_records:
            if r['wait_minutes'] is None:
                raise ValueError('Some selected patients are still waiting. Continue simulation until there are all admitted')
        return selected_records
    
    # METRIC METHODS
    def average_wait_times(self, start=0, end=None, nine_to_twelve=False, by_speciality=False):
        records = self.get_waiting_records(start, end)
        
        if nine_to_twelve:
            records = [r for r in records if 9*60 <= r['request_time'] % (24*60) < 12*60]
        
        if by_speciality:
            averages = {}

            for specialty in TreatmentType:
                waits = [r['wait_minutes'] for r in records if r['specialty']==specialty]
                averages[specialty] = np.mean(waits) if waits else float('nan')
            return averages
        return np.mean([r['wait_minutes'] for r in records]) if records else float('nan')
     
    def average_queue_length(self, start=0, end=None):
        return self.time_weighted_average(self.queue_log, start, end)

    def average_bed_utilization(self, start=0, end=None):
        average_occupied = self.time_weighted_average(self.occupied_bed_log, start, end)
        return average_occupied / self.beds.capacity

    def average_bed_unavailability(self, start=0, end=None):
        average_unavailable = self.time_weighted_average(self.unavailable_bed_log, start, end)
        return average_unavailable / self.beds.capacity

    # HELPERS

    def get_treatment_time(self, treatment_type) -> float:
        return treatment_type.treatment_time_rv.ppf(random.random())

    def get_discharge_time(self) -> float:
        return self.physician_discharge_time

    def get_cleaning_time(self) -> float:
        return self.cleaner_time_rv.ppf(random.random())

    def get_discharge_interval(self) -> tuple[float, float]:
        return self.physician_discharge_interval

    def get_free_beds(self) -> int:
        return self.beds.capacity - self.beds.count

    def get_queue_length(self) -> int:
        return len(self.beds.queue)


class Hospital(object):

    def __init__(self, env: simpy.Environment,
                 simulation_time: float,
                 ward_spec: list[tuple[int, set[TreatmentType], int]],
                 physicians_per_ward: int,
                 physician_discharge_time: float,
                 physician_discharge_interval: tuple[float, float],
                 cleaner_time_rv):
        self.env = env
        self.simulation_time = simulation_time

        # Check that all wards are included in ward spec
        # CORRECTION: One ward can have multiple specialities. 
        # Allow ward 11 and 12 to appear in lookup lists for Surgery and Cardiology
        ward_types = {treatment_type for _, treatment_types, _ in ward_spec for treatment_type in treatment_types}
        all_types = set(TreatmentType.list())
        assert all_types.issubset(ward_types), f"Ward specification does not contain treatment types {all_types.difference(ward_types)}"

        # Hospital resources
        self.wards = [Ward(env=env,
                           ward_id=ward_id,
                           treatment_type=treatment_type,
                           bed_count=bed_count,
                           physician_count=physicians_per_ward,
                           physician_discharge_time=physician_discharge_time,
                           physician_discharge_interval=physician_discharge_interval,
                           cleaner_time_rv=cleaner_time_rv)
                      for ward_id, treatment_type, bed_count in ward_spec]
        self.wards_by_type: dict[TreatmentType, list[Ward]] = {treatment_type: [] for treatment_type in TreatmentType.list()}
        for ward in self.wards:
            for treatment_type in ward.treatment_type:
                self.wards_by_type[treatment_type].append(ward)

    def average_wait_times(self, start=0, end=None, nine_to_twelve=False, by_speciality=False, by_ward=True):
        if by_ward:
            return {ward.ward_id: ward.average_wait_times(start=start, end=end, nine_to_twelve=nine_to_twelve, by_speciality=by_speciality) for ward in self.wards}

        # pool patient records
        records = [r for ward in self.wards for r in ward.get_waiting_records(start, end)]

        if nine_to_twelve:
            records = [r for r in records if (9*60<= r['request_time'] % (24*60) < 12*60)]

        if by_speciality:
            averages = {}

            for specialty in TreatmentType:
                waits = [r['wait_minutes'] for r in records if r['specialty']==specialty]
                averages[specialty] = np.mean(waits) if waits else float('nan')
            return averages

        waits = [r['wait_minutes'] for r in records]
        return np.mean(waits) if waits else float('nan')

    def average_queue_length(self, start=0, end=None, by_ward=True):
        averages_by_ward = {ward.ward_id: ward.average_queue_length(start,end) for ward in self.wards}
        if by_ward:
            return averages_by_ward
        # average total number of patients across all wards
        return sum(averages_by_ward.values())

    def average_bed_utilization(self, start=0, end=None, by_ward=True):
        averages_by_ward = {ward.ward_id: ward.average_bed_utilization(start,end) for ward in self.wards}
        if by_ward:
            return averages_by_ward
        # hospital utilization is waited by ward bed capacity
        total_beds = sum(ward.beds.capacity for ward in self.wards)
        if not total_beds:
            return float('nan')
        weighted_average = sum(averages_by_ward[ward.ward_id]*ward.beds.capacity for ward in self.wards) / total_beds
        return weighted_average if total_beds else float('nan')

    def oncology_fraction_over_5_hours(self, start=0, end=None): # Only 1 ward has Onco
        records = [r for ward in self.wards for r in ward.get_waiting_records(start, end)]
        oncology_records = [r for r in records if r['specialty'] == TreatmentType.ONCO]
        if not oncology_records:
            return float('nan')

        count_over_5 = sum([1 for r in oncology_records if r['wait_minutes'] > 5*60])
        return count_over_5 / len(oncology_records)

    def get_wards_for_type(self, treatment_type: TreatmentType) -> list[Ward]:
        return self.wards_by_type[treatment_type]

In [73]:
# Ward allocation
def get_hospital_ward(hospital: Hospital, treatment_type: TreatmentType) -> Ward:
    wards = hospital.get_wards_for_type(treatment_type).copy()
    if len(wards) == 1:
        return wards[0]
    # 1: assign to ward with the most free beds
    wards = get_wards_with_most_free_beds(wards)
    # 2: assign to ward with shortest queue
    wards = get_wards_with_shortest_queues(wards)
    # 3: select ward randomly, each equal probability
    return wards[0] if len(wards) == 1 else random.choices(wards)[0]


def get_wards_with_most_free_beds(wards) -> list[Ward]:
    return get_best_values(wards, key_func=lambda ward: ward.get_free_beds(), select=max)


def get_wards_with_shortest_queues(wards) -> list[Ward]:
    return get_best_values(wards, key_func=lambda ward: ward.get_queue_length(), select=min)


def get_best_values(li, key_func, select) -> list[typing.Any]:
    if len(li) <= 1:
        return li
    keyed_values = {}
    for x in li:
        k = key_func(x)
        if k not in keyed_values:
            keyed_values[k] = []
        keyed_values[k].append(x)
    return keyed_values[select(keyed_values)]

In [ ]:
# Process
def hospital_arrival(env, hospital):

    while True:
        # Arrival rates will change at the boundary to be consistent 
        # with the piecewise-constant hourly poisson process
        hour = int(env.now // 60) % 24
        # find the upper hour
        boundary = (int(env.now // 60) + 1) * 60
        rate = hourly_rates[hour]
        next_arrival_time = stats.expon(scale=60/rate).ppf(random.random())

        # Next hour reached, sample using its rate
        if env.now + next_arrival_time >= boundary:
            yield env.timeout(boundary-env.now)
            continue

        # Arrival occurs within the current hour
        yield env.timeout(next_arrival_time)

        treatment_type = TreatmentType.random_type()
        ward = get_hospital_ward(hospital, treatment_type)
        env.process(ward_treatment(env, ward, treatment_type))


def ward_treatment(env, ward, treatment_type):

    record = {'specialty': treatment_type, 'request_time': env.now, 'admission_time': None, 'wait_minutes': None}
    ward.patient_records.append(record)

    bed_request = ward.beds.request()
    ward.log_queue()
    yield bed_request
    ward.log_queue()
    
    record["admission_time"] = env.now
    record['wait_minutes'] = record["admission_time"] - record['request_time']

    # Bed is occupied and unavailable to other patients
    ward.num_occupied_beds += 1
    ward.num_unavailable_beds += 1
    ward.log_occupied_beds()
    ward.log_unavailable_beds()
    treatment_time = days_to_minutes(ward.get_treatment_time(treatment_type))
    yield env.timeout(treatment_time)
    yield env.process(ward_discharge(env, ward))
    # Treatment complete and patient is discharged; bed is unoccupied but still unavailable
    ward.num_occupied_beds -= 1
    ward.log_occupied_beds()
    
    yield env.process(ward_cleaning(env, ward))

    # Cleaning finishes; bed is available
    ward.num_unavailable_beds -= 1
    ward.log_unavailable_beds()

    ward.beds.release(bed_request)


def ward_discharge(env, ward):
    physician_request = ward.physicians.request()
    yield physician_request
    # Now, the physicians will handle this patient next when they can
    # Key assumption: discharge is first-come first serve;
    # the patient-processes are "waiting" but physicians will discharge
    # on basis of who finished treatment first.

    current_sim_time = env.now
    current_daytime = minute_of_day(current_sim_time)
    # Half-open interval
    discharge_interval = ward.physician_discharge_interval
    if current_daytime < discharge_interval[0]:
        # current time < 3PM --> wait until 3PM
        yield env.timeout(discharge_interval[0] - current_daytime)
    elif current_daytime >= discharge_interval[1]:
        # current time < 5PM --> wait until 3PM the next day
        yield env.timeout(discharge_interval[0] + hours_to_minutes(24) - current_daytime)

    discharge_time = ward.get_discharge_time()
    yield env.timeout(discharge_time)

    ward.physicians.release(physician_request)


def ward_cleaning(env, ward):
    # No request needed ("ample" cleaning)
    # additionally, assume cleaning runs 24/7 (unlike discharges)
    cleaning_time = ward.get_cleaning_time()
    yield env.timeout(cleaning_time)




# TODO how to interpret this statement?
# I am thinking of just considering the assigned ward's queue as being "in ED".
# -- Gabriel
# Unavailable capacity in ward -> patient waits in ED until availability

In [ ]:
def run_simulation(env, hospital, warmup_end=0):
    observation_end = hospital.simulation_time
    if not 0 <= warmup_end < observation_end:
        raise ValueError('Warm-up endpoint end before the observation endpoint')

    env.process(hospital_arrival(env, hospital))
    env.run(until=observation_end)

    # Allow all patients who are in queue and have arrived during the observation period to be admitted 
    # Otherwise waiting times tend to be underestimated
    while any(warmup_end <= record['request_time'] < observation_end and record['wait_minutes'] is None for ward in hospital.wards for record in ward.patient_records):
        env.step()

# Replications                                                                 
def run_replications(NUM_REPLICATIONS, BASE_PHYSICIANS_PER_WARD,                  
                      BASE_PHYSICIAN_DISCHARGE_TIME, BASE_PHYSICIAN_ACTIVITY_INTERVAL,                                 
                      BASE_CLEANING_INTERVAL, SIMULATION_TIME, WARMUP_END=0):                 

    T = TreatmentType
    ward_spec = [
        (1, {T.GEN_MED}, 58), (2, {T.NEURO}, 14), (3, {T.RENAL}, 27),
        (4, {T.NEURO}, 6), (5, {T.GASTRO_ENDO}, 26), (6, {T.SURG}, 34),
        (7, {T.CARD}, 24), (8, {T.ORTHO}, 70), (9, {T.ONCO}, 35),
        (10, {T.RESPI}, 14), (11, {T.SURG, T.CARD}, 24), (12, {T.SURG, T.CARD}, 24),
    ]                               

    average_wait_times_list = []                                                     
    average_queue_length_list = []                                                   
    average_bed_utilization_list = []                                               
    average_wait_times_9AM_12PM_list = [] 
    average_wait_times_9AM_12PM_by_specialty_list = []
    proportion_oncology_above_5_hours_list = []                                                      

    for replication in tqdm(range(NUM_REPLICATIONS)):                        

        env = simpy.Environment()
        hospital = Hospital(env=env, 
                            simulation_time=SIMULATION_TIME, 
                            ward_spec=ward_spec,
                            physicians_per_ward=BASE_PHYSICIANS_PER_WARD, 
                            physician_discharge_time=BASE_PHYSICIAN_DISCHARGE_TIME,
                            physician_discharge_interval=BASE_PHYSICIAN_ACTIVITY_INTERVAL, 
                            cleaner_time_rv=BASE_CLEANING_INTERVAL)        

        run_simulation(env, hospital, warmup_end=WARMUP_END)                                          

        window = {'start': WARMUP_END, 'end': SIMULATION_TIME}
        average_wait_times_list.append(hospital.average_wait_times(**window))                             
        average_queue_length_list.append(hospital.average_queue_length(**window))          
        average_bed_utilization_list.append(hospital.average_bed_utilization(**window))        
        average_wait_times_9AM_12PM_list.append(hospital.average_wait_times(**window, nine_to_twelve=True, by_ward=False))      
        average_wait_times_9AM_12PM_by_specialty_list.append(hospital.average_wait_times(**window, nine_to_twelve=True, by_speciality=True, by_ward=False))
        proportion_oncology_above_5_hours_list.append(hospital.oncology_fraction_over_5_hours(**window))
    return average_wait_times_list, average_queue_length_list, average_bed_utilization_list, average_wait_times_9AM_12PM_list, average_wait_times_9AM_12PM_by_specialty_list, proportion_oncology_above_5_hours_list 

In [ ]:
# Run simulation

# Default parameters for reference
BASE_PHYSICIANS_PER_WARD = 5
BASE_PHYSICIAN_DISCHARGE_TIME = 30 # Minutes
BASE_PHYSICIAN_ACTIVITY_INTERVAL = (hours_to_minutes(15), hours_to_minutes(17)) # [3PM, 5PM)
BASE_CLEANING_INTERVAL = stats.triang(c=0.5, loc=30, scale=30) # T ~ Triangular(30, 45, 60)
SIMULATION_TIME = days_to_minutes(100)
NUM_REPLICATIONS = 10
WARMUP_END = 0

(average_wait_times_list, average_queue_length_list, 
 average_bed_utilization_list, average_wait_times_9AM_12PM_list, 
 average_wait_times_9AM_12PM_by_specialty_list, proportion_oncology_above_5_hours_list) = run_replications(NUM_REPLICATIONS, BASE_PHYSICIANS_PER_WARD,                  
                                                                                                            BASE_PHYSICIAN_DISCHARGE_TIME, BASE_PHYSICIAN_ACTIVITY_INTERVAL,                                 
                                                                                                            BASE_CLEANING_INTERVAL, SIMULATION_TIME, WARMUP_END)  

100%|██████████| 10/10 [00:45<00:00,  4.53s/it]


In [95]:
def mean_and_95_ci(values):
    values = np.asarray(values, dtype=float)
    n = len(values)

    if n < 2:
        raise ValueError('Need at least two replications')
    if not np.all(np.isfinite(values)):
        raise ValueError('Replications contain NaN or infinity')

    mean = np.mean(values)
    standard_error = np.std(values, ddof=1) / np.sqrt(n)
    margin = standard_error * stats.t.ppf(0.975, df=n-1)
    return [mean, (round(mean-margin,2), round(mean+margin,2))]

# LONG RUN AVERAGE WAITING TIMES 
long_run_waiting_times_ward = {1: [], 2: [], 3:[], 4:[], 5: [], 6: [], 7:[], 8:[], 9: [], 10: [], 11:[], 12:[]}
for item in average_wait_times_list:
    for ward, average_time in item.items():
        long_run_waiting_times_ward[ward].append(average_time)
for ward, values in long_run_waiting_times_ward.items():
    long_run_waiting_times_ward[ward] = mean_and_95_ci(values)

# LONG RUN AVERAGE QUEUE LENGTHS
long_run_queueing_lengths_ward = {1: [], 2: [], 3:[], 4:[], 5: [], 6: [], 7:[], 8:[], 9: [], 10: [], 11:[], 12:[]}
for item in average_queue_length_list:
    for ward, average_time in item.items():
        long_run_queueing_lengths_ward[ward].append(average_time)
for ward, values in long_run_queueing_lengths_ward.items():
    long_run_queueing_lengths_ward[ward] = mean_and_95_ci(values)

# LONG RUN BED UTILIZATION
long_run_bed_utilization_ward = {1: [], 2: [], 3:[], 4:[], 5: [], 6: [], 7:[], 8:[], 9: [], 10: [], 11:[], 12:[]}
for item in average_bed_utilization_list:
    for ward, average_time in item.items():
        long_run_bed_utilization_ward[ward].append(average_time)
for ward, values in long_run_bed_utilization_ward.items():
    long_run_bed_utilization_ward[ward] = mean_and_95_ci(values)

# OVERALL LONG RUN AVERAGE WAITING TIMES. 9AM-12PM
overall_long_run_waiting_time_9AM_12PM = mean_and_95_ci(average_wait_times_9AM_12PM_list)

# LONG RUN AVERAGE WAITING TIMES. 9AM-12PM. BY SPECIALTY
T = TreatmentType
T_keys = [T.SURG, T.CARD, T.GEN_MED, T.ORTHO, T.GASTRO_ENDO, T.ONCO, T.NEURO, T.RENAL, T.RESPI]
average_wait_times_9AM_12PM_by_specialty_DICT = {specialty: [] for specialty in T_keys}
for item in average_wait_times_9AM_12PM_by_specialty_list:
    for specialty, average_time in item.items():
        average_wait_times_9AM_12PM_by_specialty_DICT[specialty].append(average_time)
for specialty, values in average_wait_times_9AM_12PM_by_specialty_DICT.items():
    average_wait_times_9AM_12PM_by_specialty_DICT[specialty] = mean_and_95_ci(values)

# LONG RUN FRACTION OF ONCOLOGY PATIENTS WITH >5h WAITING TIME
proportion_oncology_above_5_hours = mean_and_95_ci(proportion_oncology_above_5_hours_list)

In [ ]:
# Data into tables
from IPython.display import display, Markdown
def display_table(title, df):
    display(Markdown(f'### {title}'))
    display(df)

wards = [f'Ward {i+1}' for i in range(12)]
index = [f'Average of {NUM_REPLICATIONS} replications', '95% CI']

df_long_run_waiting_times_ward = pd.DataFrame(long_run_waiting_times_ward, index=index)
df_long_run_waiting_times_ward.columns = wards
display_table('Long-run average waiting times (min)',df_long_run_waiting_times_ward)

df_long_run_queueing_lengths_ward = pd.DataFrame(long_run_queueing_lengths_ward, index=index)
df_long_run_queueing_lengths_ward.columns = wards
display_table('Long-run average queue lengths (patients)', df_long_run_queueing_lengths_ward)

df_long_run_bed_utilization_ward = pd.DataFrame(long_run_bed_utilization_ward, index=index)
df_long_run_bed_utilization_ward.columns = wards
display_table('Long-run bed utilization (fraction)', df_long_run_bed_utilization_ward)

df_morning_by_specialty = pd.DataFrame(average_wait_times_9AM_12PM_by_specialty_DICT, index=index)
display_table('Average waiting times for requests arriving between 9AM-12PM (min), by specialty', df_morning_by_specialty)

index1 = [f'Overall average across 12 wards and {NUM_REPLICATIONS} replications', '95% CI']
df_overall_long_run_waiting_time_9AM_12PM = pd.DataFrame(overall_long_run_waiting_time_9AM_12PM, index=index1)
display_table('Overall average waiting time for requests arriving between (min) 9AM-12PM', df_overall_long_run_waiting_time_9AM_12PM)

df_proportion_oncology_above_5_hours = pd.DataFrame(proportion_oncology_above_5_hours, index=index1)
display_table('Fraction of Oncology patients waiting for more than 5h', df_proportion_oncology_above_5_hours)

### Long-run average waiting times (min)

,Ward 1,Ward 2,Ward 3,Ward 4,Ward 5,Ward 6,Ward 7,Ward 8,Ward 9,Ward 10,Ward 11,Ward 12
Average of 10 replications,212.209045,116.802769,8.206285,479.423387,258.600805,13.109051,25.602696,77.962193,1231.151129,358.704703,26.549794,27.656
95% CI,"(18.35, 406.07)","(61.57, 172.03)","(-0.16, 16.58)","(294.79, 664.05)","(134.74, 382.46)","(7.21, 19.01)","(13.06, 38.15)","(23.25, 132.67)","(483.91, 1978.39)","(93.08, 624.33)","(14.82, 38.28)","(14.77, 40.54)"


### Long-run average queue lengths (patients)

,Ward 1,Ward 2,Ward 3,Ward 4,Ward 5,Ward 6,Ward 7,Ward 8,Ward 9,Ward 10,Ward 11,Ward 12
Average of 10 replications,1.719184,0.26389,0.02015,0.381159,1.021899,0.092209,0.102729,0.549404,4.337412,0.713475,0.119684,0.120349
95% CI,"(0.1, 3.33)","(0.14, 0.39)","(-0.0, 0.04)","(0.23, 0.53)","(0.53, 1.51)","(0.05, 0.13)","(0.05, 0.15)","(0.16, 0.94)","(1.61, 7.07)","(0.18, 1.25)","(0.06, 0.18)","(0.06, 0.18)"


### Long-run bed utilization (fraction)

,Ward 1,Ward 2,Ward 3,Ward 4,Ward 5,Ward 6,Ward 7,Ward 8,Ward 9,Ward 10,Ward 11,Ward 12
Average of 10 replications,0.83574,0.773478,0.635801,0.661966,0.8079,0.813924,0.783251,0.787642,0.84087,0.74061,0.793347,0.794495
95% CI,"(0.82, 0.85)","(0.75, 0.8)","(0.6, 0.67)","(0.6, 0.72)","(0.78, 0.83)","(0.8, 0.83)","(0.76, 0.8)","(0.76, 0.82)","(0.8, 0.88)","(0.71, 0.77)","(0.77, 0.81)","(0.78, 0.81)"


### Average waiting time for requests arriving between 9AM-12PM

,0
Overall average across 12 wards and 10 replications,168.787693
95% CI,"(116.08, 221.5)"


### Fraction of Oncology patients waiting for more than 5h

,0
Overall average across 12 wards and 10 replications,0.385371
95% CI,"(0.2, 0.57)"


In [ ]:
# TODO: select appropriate warm-up time, observation period, simulation length, num of replications, 